In [13]:
import requests
import time
from bs4 import BeautifulSoup
from datetime import date, timedelta
from urllib.parse import urlparse
import json
import logging
import pandas as pd

In [35]:
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)s  %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger(__name__)

TAGESSCHAU_BASE = "https://www.tagesschau.de"
API_BASE        = "https://www.tagesschau.de/api2u"

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (compatible; AwesomeTagesschauScraper/1.0; "
        "+https://huggingface.co/datasets/stefan-it/awesome-tagesschau)"
    )
}
# Update request delay depending on what we need
REQUEST_DELAY = 5
RESSORT_FILTER = "ausland"

In [36]:
def generate_date_range(start: date, end: date):
    cur = start
    while cur <= end:
        yield cur
        cur += timedelta(days=1)


In [37]:
def _get(url: str, retries: int = 3) -> requests.Response | None:
    for attempt in range(retries):
        try:
            r = requests.get(url, headers=HEADERS, timeout=15)
            r.raise_for_status()
            time.sleep(REQUEST_DELAY)
            return r
        except requests.RequestException as exc:
            wait = 2 ** attempt
            log.warning("  Request failed (%s), retrying in %ds …", exc, wait)
            time.sleep(wait)
    log.error("  Giving up on %s", url)
    return None


def _extract_article_links(soup: BeautifulSoup) -> set:
    """
    Extract article links from a parsed archive page.
    """
    links = set()
    container = soup.find("div", class_="container")
    if not container:
        return links

    for a in container.find_all("a", href=True):
        href = a["href"]
        # Skip external links (e.g. DW, sportschau sub-domains)
        if not href.startswith("/"):
            continue
        # Skip non-article paths
        if any(href.startswith(p) for p in ("/archiv", "/suche", "/video", "/sendung")):
            continue
        if href.endswith(".html"):
            links.add(href)
    return links


def collect_archive_links(start: date, end: date, ressort=None) -> set:
    all_links = set()

    # Build the filter suffix once — e.g. "&filter=ausland" or "" if no filter
    filter_suffix = f"&filter={ressort}" if ressort else ""

    for d in generate_date_range(start, end):
        log.info("  Crawling archive for %s (filter: %s)", d.isoformat(), ressort or "none")
        url = f"{TAGESSCHAU_BASE}/archiv?datum={d.isoformat()}{filter_suffix}"

        r = _get(url)
        if r is None:
            log.warning("  Skipping %s — request failed", d.isoformat())
            continue

        soup = BeautifulSoup(r.text, "html.parser")
        new_links = _extract_article_links(soup)

        all_links |= new_links
        log.info("    found %d links (%d total so far)", len(new_links), len(all_links))

    return all_links

In [38]:
def fetch_article(path: str):
    """
    Fetch full article JSON from the Tagesschau API2u endpoint.
    """
    url = f"{API_BASE}{path}"
    r = _get(url)
    if r is None:
        return None
    try:
        return r.json()
    except ValueError:
        log.warning("  Non-JSON response for %s", path)
        return None

In [40]:
def main():
    start_date = date(2025, 5, 15)
    end_date   = date(2025, 5, 22)

    output_file = (
        f"tagesschau_{start_date.isoformat()}_to_{end_date.isoformat()}.json"
    )

    log.info("Collecting archive links")
    archive_links = collect_archive_links(start_date, end_date, ressort=RESSORT_FILTER)
    log.info("Total unique article paths found: %d", len(archive_links))

    log.info("Fetching article details from API")
    articles = []
    failed   = []

    for i, path in enumerate(sorted(archive_links), 1):
        log.info("  [%d/%d] %s", i, len(archive_links), path)
        data = fetch_article(path)
        if data:
            articles.append(data)
        else:
            failed.append(path)

    log.info("Done")
    log.info("Successfully fetched: %d", len(articles))
    log.info("Failed: %d", len(failed))
    if failed:
        log.info("Failed paths: %s", json.dumps(failed, ensure_ascii=False))

    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(articles, f, ensure_ascii=False, indent=2)
    log.info("Saved to %s", output_file)


if __name__ == "__main__":
    main()

13:57:51  INFO  Collecting archive links
13:57:51  INFO    Crawling archive for 2025-05-15 (filter: ausland)
13:57:59  INFO      found 16 links (16 total so far)
13:57:59  INFO    Crawling archive for 2025-05-16 (filter: ausland)
13:58:06  INFO      found 13 links (29 total so far)
13:58:06  INFO    Crawling archive for 2025-05-17 (filter: ausland)
13:58:13  INFO      found 14 links (43 total so far)
13:58:13  INFO    Crawling archive for 2025-05-18 (filter: ausland)
13:58:20  INFO      found 17 links (60 total so far)
13:58:20  INFO    Crawling archive for 2025-05-19 (filter: ausland)
13:58:29  INFO      found 20 links (80 total so far)
13:58:29  INFO    Crawling archive for 2025-05-20 (filter: ausland)
13:58:37  INFO      found 17 links (97 total so far)
13:58:37  INFO    Crawling archive for 2025-05-21 (filter: ausland)
13:58:44  INFO      found 16 links (113 total so far)
13:58:44  INFO    Crawling archive for 2025-05-22 (filter: ausland)
13:58:52  INFO      found 17 links (130 tot

In [41]:
with open("tagesschau_2025-05-15_to_2025-05-22.json", "r", encoding="utf-8") as f:
    articles = json.load(f)

df = pd.json_normalize(articles)
print(df.shape)
print(df.columns.tolist())

(130, 37)
['sophoraId', 'externalId', 'title', 'date', 'tags', 'updateCheckUrl', 'content', 'tracking', 'topline', 'firstSentence', 'images', 'details', 'detailsweb', 'shareURL', 'geotags', 'regionId', 'regionIds', 'ressort', 'breakingNews', 'type', 'teaserImage.copyright', 'teaserImage.alttext', 'teaserImage.imageVariants.1x1-144', 'teaserImage.imageVariants.1x1-256', 'teaserImage.imageVariants.1x1-432', 'teaserImage.imageVariants.1x1-640', 'teaserImage.imageVariants.1x1-840', 'teaserImage.imageVariants.16x9-256', 'teaserImage.imageVariants.16x9-384', 'teaserImage.imageVariants.16x9-512', 'teaserImage.imageVariants.16x9-640', 'teaserImage.imageVariants.16x9-960', 'teaserImage.imageVariants.16x9-1280', 'teaserImage.imageVariants.16x9-1920', 'teaserImage.type', 'comments', 'teaserImage.title']


In [43]:
pd.set_option("display.max_rows", None)
df.head(10)     

,sophoraId,externalId,title,date,tags,updateCheckUrl,content,tracking,topline,firstSentence,...,teaserImage.imageVariants.16x9-256,teaserImage.imageVariants.16x9-384,teaserImage.imageVariants.16x9-512,teaserImage.imageVariants.16x9-640,teaserImage.imageVariants.16x9-960,teaserImage.imageVariants.16x9-1280,teaserImage.imageVariants.16x9-1920,teaserImage.type,comments,teaserImage.title
0,suedafrika-trump-102,50701434-ad21-43c6-880b-ab477e0c4a99,Schwere Vorwürfe belasten Treffen im Weißen Haus,2025-05-21T06:00:57.277+02:00,"[{'tag': 'Südafrika'}, {'tag': 'Ramaphosa'}, {...",https://www.tagesschau.de/api2u/suedafrika-tru...,[{'value': '<strong>Die US-Regierung wirft Süd...,[{'sid': 'app.ausland.afrika.suedafrika-trump-...,Südafrikas Präsident bei Trump,Die USA werfen Südafrika Rassismus gegen Weiße...,...,https://images.tagesschau.de/image/3f86f5ce-13...,https://images.tagesschau.de/image/3f86f5ce-13...,https://images.tagesschau.de/image/3f86f5ce-13...,https://images.tagesschau.de/image/3f86f5ce-13...,https://images.tagesschau.de/image/3f86f5ce-13...,https://images.tagesschau.de/image/3f86f5ce-13...,https://images.tagesschau.de/image/3f86f5ce-13...,image,NaN,NaN
1,suedafrika-trump-104,6e2e2369-ab3f-421c-9c42-937ee416d403,Lob für Ramaphosas Ruhe - aber auch Kritik,2025-05-22T15:35:00.021+02:00,"[{'tag': 'USA'}, {'tag': 'Trump'}, {'tag': 'Sü...",https://www.tagesschau.de/api2u/suedafrika-tru...,[{'value': '<strong>US-Präsident Trump hat sei...,[{'sid': 'app.ausland.afrika.suedafrika-trump-...,Reaktionen in Südafrika,"Wie hat sich das ""Team Südafrika"" in Washingto...",...,https://images.tagesschau.de/image/53a58195-a6...,https://images.tagesschau.de/image/53a58195-a6...,https://images.tagesschau.de/image/53a58195-a6...,https://images.tagesschau.de/image/53a58195-a6...,https://images.tagesschau.de/image/53a58195-a6...,https://images.tagesschau.de/image/53a58195-a6...,https://images.tagesschau.de/image/53a58195-a6...,image,NaN,NaN
2,argentinien-unwetter-ueberschwemmungen-100,21a84291-8bd6-460c-b6b6-b687828a2bb4,Heftige Überschwemmungen in Argentinien,2025-05-18T12:47:27.410+02:00,"[{'tag': 'Argentinien'}, {'tag': 'Unwetter'}, ...",https://www.tagesschau.de/api2u/argentinien-un...,[{'value': '<strong>Starke Regenfälle haben in...,[{'sid': 'app.ausland.amerika.argentinien-unwe...,Tausende in Sicherheit gebracht,"Teile des Landes stehen unter Wasser, und ein ...",...,https://images.tagesschau.de/image/d2a73a8a-20...,https://images.tagesschau.de/image/d2a73a8a-20...,https://images.tagesschau.de/image/d2a73a8a-20...,https://images.tagesschau.de/image/d2a73a8a-20...,https://images.tagesschau.de/image/d2a73a8a-20...,https://images.tagesschau.de/image/d2a73a8a-20...,https://images.tagesschau.de/image/d2a73a8a-20...,image,NaN,NaN
3,biden-gesundheitsprobleme-100,6896d0ae-7188-4116-964d-3c2384d9cde9,"""Es gab einen Biden, der nicht mehr funktionie...",2025-05-16T09:14:39.610+02:00,"[{'tag': 'USA'}, {'tag': 'Biden'}, {'tag': 'Ge...",https://www.tagesschau.de/api2u/biden-gesundhe...,[{'value': '<strong>Massive geistige Aussetzer...,[{'sid': 'app.ausland.amerika.biden-gesundheit...,Buch über Gesundheitszustand,Massive geistige Aussetzer: Ein neues Buch zei...,...,https://images.tagesschau.de/image/c01d9a32-a0...,https://images.tagesschau.de/image/c01d9a32-a0...,https://images.tagesschau.de/image/c01d9a32-a0...,https://images.tagesschau.de/image/c01d9a32-a0...,https://images.tagesschau.de/image/c01d9a32-a0...,https://images.tagesschau.de/image/c01d9a32-a0...,https://images.tagesschau.de/image/c01d9a32-a0...,image,NaN,NaN
4,biden-krebs-100,3fcaf421-f0d8-4a71-855e-a0b85116b3de,Biden an Prostatakrebs erkrankt,2025-05-19T04:10:01.712+02:00,"[{'tag': 'Biden'}, {'tag': 'USA'}]",https://www.tagesschau.de/api2u/biden-krebs-10...,[{'value': '<strong>Der ehemalige US-Präsident...,[{'sid': 'app.ausland.amerika.biden-krebs-100'...,Früherer US-Präsident,Nach Angaben seines Büros handelt es sich um e...,...,https://images.tagesschau.de/image/77206bba-c5...,https

In [44]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 130 entries, 0 to 129
Data columns (total 37 columns):
 #   Column                               Non-Null Count  Dtype 
---  ------                               --------------  ----- 
 0   sophoraId                            130 non-null    str   
 1   externalId                           130 non-null    str   
 2   title                                130 non-null    str   
 3   date                                 130 non-null    str   
 4   tags                                 130 non-null    object
 5   updateCheckUrl                       130 non-null    str   
 6   content                              130 non-null    object
 7   tracking                             130 non-null    object
 8   topline                              130 non-null    str   
 9   firstSentence                        130 non-null    str   
 10  images                               130 non-null    object
 11  details                              130 non-null    str